# Train, then evaluate: a chain with lineage

Two AI Jobs that feed each other, from a notebook: a LoRA fine-tune, then an evaluation of the adapter
it produced against its base model, on the records the training held out. The SDK's **chain** runs
them in order, hands the adapter from one to the next, and records the lineage: what each step
consumed and produced, by content hash, from the base model's commit and the dataset's to the
adapter and the evaluation.

The chain lives in the cluster (labels and annotations on the runs' AIJobs), so it can be read back
after a restart, and a step whose inputs have not changed is not run again: its earlier run is
reused.

**Needs:** AI Factory with the AIJob API, a training profile that takes a script and a dataset volume
and keeps a checkpoint volume (`shared-gpu-dev` here: a 4 GiB GPU share), and Application Collection
credentials in AI Factory's Settings.

## 1. The SDK

In [ ]:
%pip install -q "rancher-ai[pandas] @ https://github.com/dstanley/aif/archive/refs/heads/pr/7-designs.tar.gz#subdirectory=sdk/python"

## 2. Settings

In [ ]:
CONTEXT = None                 # kubeconfig context of the cluster (None: the current one)
KUBECONFIG = None              # a kubeconfig file (None: $KUBECONFIG or ~/.kube/config)
PROJECT = "ai-data-science"    # the project's namespace
PROFILE = "shared-gpu-dev"     # takes a script and a dataset volume, keeps a checkpoint volume
CHAIN = "lora-chain-demo"      # the chain's name: its runs are labelled with it

TRAIN_ENV = {"MODEL": "Qwen/Qwen2.5-0.5B-Instruct", "STEPS": "60", "BATCH": "4", "SEQ_LEN": "256",
             "LOG_EVERY": "10", "HELD_OUT": "32", "LORA_R": "16", "LR": "2e-4"}

## 3. The scripts

The training script is the LoRA Distributed Training Test's, which reports its artifacts
(`AIF_ARTIFACT` lines: the base model at its commit, the dataset and the adapter with their hashes);
the evaluation script takes the adapter as `AIF_INPUT_ADAPTER`. Both from the aif-lab repository.

In [ ]:
import os, urllib.request
from rancher_ai import Client

ai = Client(context=CONTEXT, project=PROJECT, kubeconfig=KUBECONFIG)
SCRIPTS = "https://raw.githubusercontent.com/dstanley/aif-lab/main/profile-packs/nvidia-180g/scripts/"

def script(name):
    local = os.path.join("..", "profile-packs", "nvidia-180g", "scripts", name)   # in an aif-lab checkout
    if os.path.exists(local):
        return open(local).read()
    return urllib.request.urlopen(SCRIPTS + name).read().decode()

train_py, eval_py = script("lora_distributed_test.py"), script("lora_eval_adapter.py")
chain = ai.chain(CHAIN)
chain

## 4. Train

The first step: `chain.run(step, profile, ...)` submits the run, waits for it, and keeps the
artifacts it reported on its record.

In [ ]:
train = chain.run("train", PROFILE, script=train_py, env=TRAIN_ENV)
train.output("adapter")

## 5. Evaluate the adapter

The second step takes the training run's adapter as an input: the chain mounts the volume it is on
(read-only) and hands the script its path. It also takes the dataset the training recorded, so the
evaluation reads the same revision of it.

In [ ]:
evals = chain.run("eval", PROFILE, script=eval_py,
                  inputs={"adapter": train.output("adapter"), "dataset": train.input("dataset")})
evals.result()

## 6. The chain: steps and artifacts

One row per step, with its times and metrics; then one row per artifact, with which step produced
what a later step consumed (matched by content hash, not by name).

In [ ]:
chain.frame()

In [ ]:
chain.artifacts()

## 7. Unchanged steps are not run again

Running the same step with the same profile, fields and inputs returns the run that already
succeeded, while its outputs are still there: the key is the values the run would install (image,
script, environment, seed), the training chart's version and the inputs by content. Change any of
them (or pass `cache=False`) to run it again. A step still running with the same key is waited on,
so a notebook restarted mid-chain picks up where it was.

In [ ]:
again = chain.run("train", PROFILE, script=train_py, env=TRAIN_ENV)
again.name == train.name, again.cached

## Where this goes next

This is the interactive end of a pipeline: a data scientist's chain of runs, with lineage, from a
notebook. Each step is still an AIJob, so the project's quota, the profile's limits and the run record
apply. For scheduled retraining and fixed production workflows, the same steps fit Kubeflow
Pipelines: a pipeline component that calls `chain.run` (or `runs.create`) submits each step as an AIJob
and passes on its metrics and artifact URIs.

Clean up with **Data** (the runs' volumes) and **Jobs** in the cluster's AI Jobs, or
`ai.runs.get(name).delete()`.